In [ ]:
#| default_exp helper.tests

In [ ]:
#| export
import os
from os import PathLike
from pathlib import Path
from unittest.mock import MagicMock

from bs4 import BeautifulSoup


In [ ]:
from nbdev import show_doc

In [ ]:
#| export
def _test_directory() -> Path:
    """Returns the `nbs/_tests` directory of the `trouver` repository.
    
    Assumes that the current working directory is either the root of the
    repository or the `nbs` folder and the `nbs/_tests` folder exists in the
    repository but the root of the repository does not have a folder named
    `_tests`.
    """
    cwd = os.getcwd()
    cwd_name = os.path.basename(cwd)
    if cwd_name == 'nbs':
        return Path(cwd) / '_tests'
    else:
        return Path(cwd) / 'nbs'/ '_tests'

In [ ]:
show_doc(_test_directory)

---

[source](https://github.com/hyunjongkimmath/trouver/blob/main/trouver/helper/tests.py#L11){target="_blank" style="float:right; font-size:smaller"}

### _test_directory

>      _test_directory ()

Returns the `nbs/_tests` directory of the `trouver` repository.

Assumes that the current working directory is either the root of the
repository or the `nbs` folder and the `nbs/_tests` folder exists in the
repository but the root of the repository does not have a folder named
`_tests`.

Test files for `trouver` are contained in `nbs/_tests` directory within the repository. The hidden `_test_directory()` method returns this directory.

In [ ]:
assert os.path.exists(_test_directory())

In [ ]:
os.listdir(_test_directory())

['empty_model_vault',
 'latex_examples',
 'ml_examples',
 'test_vault_1',
 'test_vault_10',
 'test_vault_2',
 'test_vault_3',
 'test_vault_4',
 'test_vault_5',
 'test_vault_6',
 'test_vault_7',
 'test_vault_8',
 'test_vault_9']

In [ ]:
#| export
_EXAMPLE_VAULT_NOTE_TEXT = r"""---
cssclass: clean-embeds
aliases: []
tags: [_meta/concept, _meta/literature_note]
---
# Grothendieck-Witt ring elements of a finite field are given by rank and discriminant up to squares[^1]

$\operatorname{GW}(\mathbb{F}_q)\cong \mathbb{Z}\times \mathbb{F}_q^{\times}/(\mathbb{F}_q^{\times})^2$[^2] where the isomorphism is given by the rank and discriminant.


[^2]: ![[notation_GW_k_Grothendieck_Witt_ring_of_a_field]]


# See Also
- [[grothendieck_witt_ring_of_a_polynomial_ring_over_a_field_is_isomorphic_to_that_of_the_field]]

# Meta
## References
![[_reference_pauli_wickelgren_aa1]]


## Citations and Footnotes
[^1]: Pauli, Wickelgren, Example 3.7, Page 4"""


def make_example_vault(temp_dir: PathLike):
    """
    Build the example reference-folder vault tree used by `obsidian` tests.

    Creates `algebraic_geometry/a1_homotopy_theory/pauli_wickelgren_aa1/
    3_the_grothendieck_witt_ring_of_k/` with index notes at each level and
    one example note whose text is `_EXAMPLE_VAULT_NOTE_TEXT`.
    """
    temp_dir = Path(temp_dir)
    os.mkdir(temp_dir / 'algebraic_geometry')
    os.mkdir(temp_dir / 'algebraic_geometry' / 'a1_homotopy_theory')
    os.mkdir(temp_dir / 'algebraic_geometry' / 'a1_homotopy_theory' / 'pauli_wickelgren_aa1')
    os.mkdir(temp_dir / 'algebraic_geometry' / 'a1_homotopy_theory' / 'pauli_wickelgren_aa1' / '3_the_grothendieck_witt_ring_of_k')

    (temp_dir / 'algebraic_geometry' / '_index_algebraic_geometry.md').touch()
    (temp_dir / 'algebraic_geometry' / 'a1_homotopy_theory' / '_index_a1_homotopy_theory.md').touch()
    (temp_dir / 'algebraic_geometry' / 'a1_homotopy_theory' / 'pauli_wickelgren_aa1' / '_index_pauli_wickelgren_aa1.md').touch()
    (temp_dir / 'algebraic_geometry' / 'a1_homotopy_theory' / 'pauli_wickelgren_aa1' / '3_the_grothendieck_witt_ring_of_k' / 'pauli_wickelgren_aa1_example 3.7.md').touch()

    with open((temp_dir / 'algebraic_geometry' / 'a1_homotopy_theory' / 'pauli_wickelgren_aa1' / '3_the_grothendieck_witt_ring_of_k' / 'pauli_wickelgren_aa1_example 3.7.md'), 'w') as writer:
        writer.write(
            _EXAMPLE_VAULT_NOTE_TEXT)


In [ ]:
#| export
class MockTokenizer:
    """
    Test double for a HuggingFace tokenizer.

    Returns a fixed-length `input_ids` list so pipeline callers can
    estimate token counts without loading a real tokenizer.
    """
    model_max_length = 512

    def __call__(self, text):
        length = len(text) // 5 + 1
        return {'input_ids': [0] * length}


class MockPipeline:
    """
    Test double for `TokenClassificationPipeline`.

    Returns the `preds` given at construction for any input text; its
    `tokenizer` is a `MockTokenizer`. Shared by token-classification
    notebook tests.
    """

    def __init__(self, preds):
        self.tokenizer = MockTokenizer()
        self._preds = preds

    def __call__(self, text):
        return self._preds


In [ ]:
#| export
def _make_bs4_tag(html_str: str):
    """Return the BeautifulSoup `Tag` parsed from `html_str`; test helper."""
    return BeautifulSoup(html_str, 'html.parser').find()


def _mock_note_data(name, reverse_count=0, direct_links=None, tags=None, notation_str=""):
    """
    Return a `MagicMock` shaped like `NoteData` for note-linking tests.

    Superset of the two historical copies in the dataset-sieve and
    data-augmentation notebooks: `data_string` and `deepcopy` are pinned so
    augmentation code paths work as well.
    """
    m = MagicMock()
    m.note_name = name
    m.data_string.return_value = f"Mock Data for {name}"
    m.deepcopy.return_value = m
    m.reverse_linked_notes = {f"in_{i}" for i in range(reverse_count)}
    m.directly_linked_notes = direct_links if direct_links else {}
    m.tags = tags
    m.parsed = MagicMock()
    m.parsed.notation_str = notation_str if notation_str else name
    return m
